In [ ]:
from src.analytics.observability_assets import write_observability_tables


def get_pipeline_id():
    """Fetch pipeline ID automatically if notebook is attached to a DLT pipeline."""
    try:
        return spark.conf.get("pipelines.pipelineId")
    except Exception:
        return None


dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("analytics_schema", "analytics", "Analytics schema")
dbutils.widgets.text("pipeline_id", "", "Pipeline ID (optional if attached to pipeline)")
dbutils.widgets.text("published_event_log_table", "", "Published event-log table")

CATALOG = dbutils.widgets.get("catalog").strip()
ANALYTICS_SCHEMA = dbutils.widgets.get("analytics_schema").strip()
WIDGET_PIPELINE_ID = dbutils.widgets.get("pipeline_id").strip()
PUBLISHED_EVENT_LOG_TABLE = dbutils.widgets.get("published_event_log_table").strip()

AUTO_PIPELINE_ID = get_pipeline_id()
PIPELINE_ID = WIDGET_PIPELINE_ID or AUTO_PIPELINE_ID

if AUTO_PIPELINE_ID:
    print(f"✓ Auto-detected pipeline ID: {AUTO_PIPELINE_ID}")
elif WIDGET_PIPELINE_ID:
    print(f"✓ Using widget pipeline ID: {WIDGET_PIPELINE_ID}")

if PIPELINE_ID and PUBLISHED_EVENT_LOG_TABLE:
    raise ValueError("Provide only one of pipeline_id or published_event_log_table.")
if not PIPELINE_ID and not PUBLISHED_EVENT_LOG_TABLE:
    raise ValueError("Provide either pipeline_id or published_event_log_table. If running in a DLT pipeline notebook, the pipeline_id is auto-detected.")

print(f"Building observability assets into {CATALOG}.{ANALYTICS_SCHEMA}")

In [ ]:
persisted_tables = write_observability_tables(
    spark,
    pipeline_id=PIPELINE_ID or None,
    published_event_log_table=PUBLISHED_EVENT_LOG_TABLE or None,
    catalog=CATALOG,
    analytics_schema=ANALYTICS_SCHEMA,
)

display(
    spark.createDataFrame(
        [{"table_name": table_name, "status": status} for table_name, status in persisted_tables.items()]
    )
)

In [ ]:
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_pipeline_updates"))
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_expectation_metrics"))
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_user_actions"))
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_latest_failures"))